# Teacher model: jitter and volume augmentation

**Question.** Does a new kind of augmentation, **jitter**, help the acted-only teacher, alone or
together with **volume**?

| Method | What one copy of a window gets |
|---|---|
| `none` | no copies (baseline) |
| `volume` | one multiplier for the whole window: target face size 0.8–5.5, kept in ×1/3–×3 (model C's rule) |
| `jitter` | a different multiplier for **every AU in every frame**, uniform in ×0.8–×1.2 |
| `volume+jitter` | volume first, then jitter |

**Both corpora:**
- `with_emojihero`: old acted + EmojiHero + new acted (model D's exact data, 8,181 windows);
- `without_emojihero`: old acted + new acted (2,397 windows).

**Same for every run:**
- "add" mode: every epoch shows all original windows plus one fresh copy of each (+100%);
- model D's architecture and training: SGD 1e-3, batch 10, at most 30 epochs, early stopping
  (patience 8) on validation;
- **5 seeds per setting**, because one run is not enough (seed noise is about 0.02 macro-F1);
- the test sets are looked at only after training, with gate temperature 0.7 (as deployed).

**Steps.** 1 Setup → 2 Settings → 3 Data → 4 Two corpora → 5 Model → 6 Augmentation →
7 Check the augmentation → 8 Training and evaluation → 9 Run all → 10 Results.

Runs are saved to Drive one by one. If Colab disconnects, run steps 1–8 again and then step 9:
finished runs are skipped.

## 1. Setup

In [ ]:
# ============================================================
# 1. SETUP: Google Drive and libraries
# ============================================================

import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import ttest_rel
from sklearn.metrics import f1_score, log_loss, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Settings
Change only this cell.

In [ ]:
# ============================================================
# 2. SETTINGS
# ============================================================

# The exact training/validation/test arrays of model D, exported from the volume-augmentation
# notebook. Upload the file to Drive and put its path here.
DATA_FILE = Path(
    "/content/drive/MyDrive/outputs_15_participants_70AU-NOT_normalize_teacher_data-self_filtered_"
    "NATURAL_ONLY-EmojiHero_Fixed-Only_Acted_data_training-Volume_Augmentation_with_EmojiHero_V2/"
    "teacher_arrays_with_emojihero.npz"
)

# Models, training logs and result tables are written to the same folder as the data file.
OUT_DIR = DATA_FILE.parent

CORPORA = ["with_emojihero", "without_emojihero"]
METHODS = ["none", "volume", "jitter", "volume+jitter"]
SEEDS = [42, 1, 7, 123, 2024]

COPY_FRACTION = 1.0                 # +100%: one copy of every window, new copies every epoch

# Jitter: one multiplier per AU per frame
JITTER_LOW, JITTER_HIGH = 0.8, 1.2

# Volume: model C's "strong" rule
VOLUME_TARGET_LOW, VOLUME_TARGET_HIGH = 0.8, 5.5
VOLUME_GAIN_MIN, VOLUME_GAIN_MAX = 1 / 3, 3.0
NEUTRAL_GAIN_LOW, NEUTRAL_GAIN_HIGH = 0.5, 1.5
NEUTRAL_MAX_AU = 0.25               # a Neutral copy never has an AU above this

# Training (model D's settings)
EPOCHS = 30
PATIENCE = 8
BATCH_SIZE = 10
EVAL_GATE_TAU = 0.7                 # deployed value; not stored in .keras files

QUICK_CHECK = False                 # True = 1 epoch, 1 seed, results in a separate folder

if QUICK_CHECK:
    EPOCHS, SEEDS = 1, [42]
    OUT_DIR = OUT_DIR / "jitter_volume_quick_check"

# Same layout as the folder: models in models/, result tables in predictions/.
# All file names start with "jitter_volume", so no older file is overwritten.
MODELS_DIR = OUT_DIR / "models"
PREDICTIONS_DIR = OUT_DIR / "predictions"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = PREDICTIONS_DIR / "jitter_volume_results.csv"
EMOTION_FILE = PREDICTIONS_DIR / "jitter_volume_results_per_emotion.csv"

print("Data:   ", DATA_FILE, "exists" if DATA_FILE.exists() else "MISSING")
print("Output: ", OUT_DIR)
print("Runs:   ", len(CORPORA) * len(METHODS) * len(SEEDS))

## 3. Data
Load model D's arrays and check that they are the expected ones.

In [ ]:
# ============================================================
# 3. LOAD DATA
# ============================================================

Z = np.load(DATA_FILE, allow_pickle=False)

CLASS_NAMES = [str(c) for c in Z["class_names"]]
AU_NAMES = [str(a) for a in Z["au_names"]]
NEUTRAL_ID = CLASS_NAMES.index("Neutral")
CLASS_LABELS = np.arange(len(CLASS_NAMES))
SEQ_LENGTH, N_AU = Z["X_train"].shape[1:]
N_FACE_AU = 63                      # the first 63 AUs; the last 7 are tongue
SOURCE_NAMES = {0: "old acted", 1: "EmojiHero", 2: "new acted"}
SOURCE_EMOJIHERO = 1

TEST = {
    "old VR": (Z["X_test_old"], Z["Y_test_old"]),
    "new VR": (Z["X_test_new"], Z["Y_test_new"]),
    "EmojiHero": (Z["X_test_eh"], Z["Y_test_eh"]),
}

# The expected sizes (model D)
assert Z["X_train"].shape == (8181, 5, 70)
assert np.bincount(Z["S_train"]).tolist() == [1553, 5784, 844]
assert [len(Z[k]) for k in ("Y_val_old", "Y_val_new", "Y_val_eh")] == [167, 111, 2310]
assert [len(v[1]) for v in TEST.values()] == [303, 304, 2268]
assert CLASS_NAMES == ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
assert np.isclose(Z["W_train"].mean(), 1.0, atol=1e-4)
for k in Z.files:
    if k.startswith("X_"):
        assert np.isfinite(Z[k]).all() and Z[k].min() >= 0 and Z[k].max() <= 1, k

print("Classes:", CLASS_NAMES)
print("Training windows by source:",
      {SOURCE_NAMES[s]: int(n) for s, n in enumerate(np.bincount(Z["S_train"]))})
print("Test windows:", {k: len(v[1]) for k, v in TEST.items()})

## 4. Two corpora
- `with_emojihero`: model D's data and model D's sample weights, unchanged.
- `without_emojihero`: EmojiHero windows removed. The sample weights are recomputed as in the
  model A/B cell: class-balanced × source-balanced (each source gives half of the loss).
  Validation uses old and new acted only.

In [ ]:
# ============================================================
# 4. BUILD THE TWO CORPORA
# ============================================================

def class_and_source_weights(y, sources):
    # class-balanced x source-balanced, mean 1 (same as the notebook's weight cells)
    classes = np.unique(y)
    assert set(classes) == set(CLASS_LABELS)
    class_w = dict(zip(classes, compute_class_weight("balanced", classes=classes, y=y)))
    source_ids, counts = np.unique(sources, return_counts=True)
    source_w = dict(zip(source_ids, len(sources) / (len(source_ids) * counts)))
    w = np.array([source_w[s] * class_w[c] for s, c in zip(sources, y)], dtype=np.float32)
    return w / w.mean()


keep = Z["S_train"] != SOURCE_EMOJIHERO

CORPUS = {
    "with_emojihero": {
        "X": Z["X_train"], "y": Z["Y_train"], "w": Z["W_train"], "sources": Z["S_train"],
        "val": {"old_acted": (Z["X_val_old"], Z["Y_val_old"]),
                "new_acted": (Z["X_val_new"], Z["Y_val_new"]),
                "emojihero": (Z["X_val_eh"], Z["Y_val_eh"])},
    },
    "without_emojihero": {
        "X": Z["X_train"][keep], "y": Z["Y_train"][keep], "sources": Z["S_train"][keep],
        "w": class_and_source_weights(Z["Y_train"][keep], Z["S_train"][keep]),
        "val": {"old_acted": (Z["X_val_old"], Z["Y_val_old"]),
                "new_acted": (Z["X_val_new"], Z["Y_val_new"])},
    },
}

# Check: recomputing the weights for the full corpus gives model D's weights exactly
assert np.allclose(class_and_source_weights(Z["Y_train"], Z["S_train"]), Z["W_train"], atol=1e-5)
assert len(CORPUS["without_emojihero"]["X"]) == 2397

for name, c in CORPUS.items():
    print(f"\n{name}: {len(c['X'])} training windows, validation = {list(c['val'])}")
    for s in np.unique(c["sources"]):
        m = c["sources"] == s
        print(f"  {SOURCE_NAMES[s]:10s} n={m.sum():5d}  share of the loss={c['w'][m].sum() / c['w'].sum():.3f}")

## 5. Model
Model D's architecture, copied unchanged from the volume-augmentation notebook.

In [ ]:
# ============================================================
# 5. MODEL ARCHITECTURE (unchanged)
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K

# ---- TCN expert blocks ----
def _tcn_block(x, filters, kernel_size, dilation_rate, dropout=0.2, name=None):
    res = x
    y = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate, name=None if name is None else name+"_c1")(x)
    y = layers.BatchNormalization(name=None if name is None else name+"_bn1")(y)
    y = layers.ReLU(name=None if name is None else name+"_relu1")(y)
    y = layers.Dropout(dropout, name=None if name is None else name+"_do1")(y)
    y = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation_rate, name=None if name is None else name+"_c2")(y)
    y = layers.BatchNormalization(name=None if name is None else name+"_bn2")(y)
    if res.shape[-1] != filters:
        res = layers.Conv1D(filters, 1, padding="same", name=None if name is None else name+"_proj")(res)
        res = layers.BatchNormalization(name=None if name is None else name+"_proj_bn")(res)
    y = layers.Add(name=None if name is None else name+"_add")([y, res])
    y = layers.ReLU(name=None if name is None else name+"_relu2")(y)
    return y

def build_expert_tcn(x, name_prefix="exp"):
    y = _tcn_block(x, 128, 3, 1, 0.2, name=name_prefix+"_tcn1")
    y = _tcn_block(y, 128, 3, 2, 0.2, name=name_prefix+"_tcn2")
    y = _tcn_block(y, 128, 3, 4, 0.2, name=name_prefix+"_tcn3")
    y = layers.GlobalAveragePooling1D(name=name_prefix+"_gap")(y)
    return y  # (B, F)

# ---- Top-2 gate as a Keras Layer ----
class Top2Gate(layers.Layer):
    def __init__(self, num_experts=3, gate_hidden=128, tau_init=2.0,
                 lb_coef=0.02, ent_coef=0.01, name="gate", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_experts = num_experts
        self.gate_hidden = gate_hidden
        self.tau = tf.Variable(tau_init, dtype=tf.float32, trainable=False, name=f"{name}_tau")
        self.lb_coef = lb_coef
        self.ent_coef = ent_coef
        self.d1 = layers.Dense(gate_hidden, activation="relu", name=f"{name}_dense1")
        self.d2 = layers.Dense(num_experts, activation=None, name=f"{name}_logits")

    def call(self, gate_in, training=None):
        logits = self.d2(self.d1(gate_in))           # (B, E)
        w_soft = tf.nn.softmax(logits / self.tau, -1)  # (B, E)

        top2 = tf.math.top_k(w_soft, k=2)
        idx = top2.indices            # (B, 2)
        vals = tf.nn.softmax(top2.values, -1)  # re-norm top-2

        B = tf.shape(w_soft)[0]; E = tf.shape(w_soft)[1]
        row_ids = tf.repeat(tf.range(B), repeats=2)
        col_ids = tf.reshape(idx, (-1,))
        scatter_idx = tf.stack([row_ids, col_ids], axis=1)   # (B*2, 2)
        scatter_vals = tf.reshape(vals, (-1,))
        w_sparse = tf.scatter_nd(scatter_idx, scatter_vals, (B, self.num_experts))  # (B,E)

        # Load-balance loss (uniform usage)
        mean_usage = tf.reduce_mean(w_sparse, axis=0)
        uniform = tf.fill([self.num_experts], 1.0 / tf.cast(self.num_experts, tf.float32))
        lb_loss = tf.reduce_sum(tf.square(mean_usage - uniform))
        self.add_loss(self.lb_coef * lb_loss)

        # Entropy reg on soft weights (encourage exploration early)
        ent = -tf.reduce_mean(tf.reduce_sum(w_soft * tf.math.log(tf.clip_by_value(w_soft, 1e-8, 1.0)), axis=-1))
        self.add_loss(self.ent_coef * (-ent))  # minimize negative entropy

        return w_sparse, w_soft  # sparse mixture, full mixture (for diagnostics)

# ---- Temperature scheduler ----
class GateTemperatureScheduler(callbacks.Callback):
    def __init__(self, gate_layer, start_tau=2.0, end_tau=0.7, anneal_epochs=20):
        super().__init__()
        self.gate = gate_layer
        self.start_tau = float(start_tau)
        self.end_tau = float(end_tau)
        self.anneal_epochs = max(1, int(anneal_epochs))
    def on_epoch_begin(self, epoch, logs=None):
        p = min(1.0, epoch / self.anneal_epochs)
        tau = self.start_tau + p * (self.end_tau - self.start_tau)
        self.gate.tau.assign(tau)
        K.set_value(self.gate.tau, tau)
        if epoch % 5 == 0:
            print(f"[GateTemperatureScheduler] epoch={epoch} tau={tau:.3f}")

# ---- Single-input MoE builder (no domain hint) ----
def build_moe_single_input(
    input_shape, num_classes,
    tau_init=2.0, tau_final=0.7,
    gate_hidden=128, classifier_hidden=256,
    lb_coef=0.02, ent_coef=0.01,
    name="MoE_SingleInput"
):
    seq_in = layers.Input(shape=input_shape, name="seq")

    # Experts
    exp_nat = build_expert_tcn(seq_in, "exp_nat")
    exp_act = build_expert_tcn(seq_in, "exp_act")
    exp_shared = build_expert_tcn(seq_in, "exp_shared")

    # Stack experts into (B, E=3, F) using Lambda + Concatenate (avoid tf.stack on KerasTensors)
    e1 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_nat_expand")(exp_nat)       # (B,1,F)
    e2 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_act_expand")(exp_act)       # (B,1,F)
    e3 = layers.Lambda(lambda z: tf.expand_dims(z, axis=1), name="exp_share_expand")(exp_shared)  # (B,1,F)
    E_stack = layers.Concatenate(axis=1, name="experts_stack")([e1, e2, e3])  # (B,3,F)

    # Gate input: pooled sequence only (single input mode)
    gate_in = layers.GlobalAveragePooling1D(name="gate_pool")(seq_in)

    gate_layer = Top2Gate(num_experts=3, gate_hidden=gate_hidden,
                          tau_init=tau_init, lb_coef=lb_coef, ent_coef=ent_coef, name="gate")
    w_sparse, w_soft = gate_layer(gate_in)   # (B,3), (B,3)

    # Weighted sum: (B,3,1) * (B,3,F) -> sum over experts -> (B,F)
    w_expanded = layers.Lambda(lambda z: tf.expand_dims(z, -1), name="w_expand")(w_sparse)  # (B,3,1)
    fused = layers.Lambda(lambda zs: tf.reduce_sum(zs[0] * zs[1], axis=1), name="fuse")([w_expanded, E_stack])  # (B,F)

    x = layers.Dense(classifier_hidden, activation="relu", name="clf_dense")(fused)
    x = layers.Dropout(0.3, name="clf_do")(x)
    y_out = layers.Dense(num_classes, activation="softmax", name="y")(x)

    model = models.Model(seq_in, y_out, name=name)
    model.compile(optimizer=tf.keras.optimizers.SGD(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Provide a temperature scheduler that targets this gate instance
    anneal_cb = GateTemperatureScheduler(gate_layer, start_tau=tau_init, end_tau=tau_final, anneal_epochs=20)
    return model, anneal_cb

## 6. Augmentation
- **volume**: one multiplier per window (model C's rule).
- **jitter**: one multiplier per AU per frame, uniform in [0.8, 1.2].
- **volume+jitter**: volume first, then jitter.

After the multipliers, values are clipped to [0, 1]. A Neutral copy is then scaled down if needed,
so its largest AU stays at or below 0.25 (the same rule as model C's volume augmentation).

In [ ]:
# ============================================================
# 6. AUGMENTATION
# ============================================================

def face_size(X):
    # mean over the frames of the sum of the 63 face AUs
    return X[..., :N_FACE_AU].sum(-1).mean(-1)


def volume_gains(X, y, rng):
    # one multiplier per window, log-uniform, aimed at a face size of 0.8-5.5
    size = np.maximum(face_size(X), 1e-6)
    low = np.clip(VOLUME_TARGET_LOW / size, VOLUME_GAIN_MIN, VOLUME_GAIN_MAX)
    high = np.clip(VOLUME_TARGET_HIGH / size, VOLUME_GAIN_MIN, VOLUME_GAIN_MAX)
    neutral = y == NEUTRAL_ID
    low[neutral], high[neutral] = NEUTRAL_GAIN_LOW, NEUTRAL_GAIN_HIGH
    return np.exp(rng.uniform(np.log(low), np.log(high))).astype(np.float32)


def jitter_gains(X, rng):
    # one multiplier per AU per frame
    return rng.uniform(JITTER_LOW, JITTER_HIGH, size=X.shape).astype(np.float32)


def make_copies(X, y, method, rng):
    # returns augmented copies of the windows X (labels y are unchanged)
    out = X.astype(np.float32, copy=True)
    if "volume" in method:
        out *= volume_gains(X, y, rng)[:, None, None]
    if "jitter" in method:
        out *= jitter_gains(X, rng)
    out = np.clip(out, 0.0, 1.0)

    neutral = y == NEUTRAL_ID
    if neutral.any():
        peak = out[neutral].max(axis=(1, 2))
        scale = np.minimum(1.0, NEUTRAL_MAX_AU / np.maximum(peak, 1e-6))
        out[neutral] *= scale[:, None, None]
    return out


class AddCopies(tf.keras.utils.PyDataset):
    # every epoch: all original windows + fresh copies of COPY_FRACTION of them, shuffled

    def __init__(self, X, y, w, method, fraction, batch_size, seed, **kwargs):
        super().__init__(**kwargs)
        self.X, self.y, self.w = X.astype(np.float32), y.astype(np.int64), w.astype(np.float32)
        self.method = method
        self.n_copies = 0 if method == "none" else int(round(fraction * len(X)))
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self._new_epoch()

    def _new_epoch(self):
        n = len(self.X)
        chosen = np.sort(self.rng.choice(n, size=self.n_copies, replace=False))
        X_all = np.concatenate([self.X, make_copies(self.X[chosen], self.y[chosen], self.method, self.rng)])
        rows = np.concatenate([np.arange(n), chosen])
        order = self.rng.permutation(len(rows))
        self.X_epoch, self.y_epoch, self.w_epoch = X_all[order], self.y[rows][order], self.w[rows][order]

    def __len__(self):
        return math.ceil(len(self.X_epoch) / self.batch_size)

    def __getitem__(self, i):
        b = slice(i * self.batch_size, (i + 1) * self.batch_size)
        return self.X_epoch[b], self.y_epoch[b], self.w_epoch[b]

    def on_epoch_end(self):
        self._new_epoch()

## 7. Check the augmentation
Before training: how much does each method change the windows? Shape change = cosine similarity
between a copy and its original (1 = the same shape). Size change = copy face size ÷ original face
size.

In [ ]:
# ============================================================
# 7. CHECK THE AUGMENTATION (no training)
# ============================================================

def cosine(a, b):
    a, b = a.reshape(len(a), -1), b.reshape(len(b), -1)
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


check_rng = np.random.default_rng(0)
X_chk, y_chk = Z["X_train"], Z["Y_train"]
expressive = (y_chk != NEUTRAL_ID) & (face_size(X_chk) > 0.1)

rows = []
for method in METHODS[1:]:
    copies = make_copies(X_chk, y_chk, method, check_rng)
    ratio = face_size(copies[expressive]) / face_size(X_chk[expressive])
    rows.append({
        "method": method,
        "shape similarity (mean cosine)": cosine(X_chk, copies).mean(),
        "size ratio: median": np.median(ratio),
        "size ratio: 5%": np.percentile(ratio, 5),
        "size ratio: 95%": np.percentile(ratio, 95),
        "Neutral copies max AU": copies[y_chk == NEUTRAL_ID].max(),
        "values in [0, 1]": bool(copies.min() >= 0 and copies.max() <= 1),
    })
display(pd.DataFrame(rows).round(3))

# One example window (Anger), first 2 frames, the 6 AUs that move most
i = int(np.flatnonzero((y_chk == CLASS_NAMES.index("Anger")) & expressive)[0])
top = np.argsort(-X_chk[i].mean(0))[:6]
example = {"original": X_chk[i:i + 1]}
for method in METHODS[1:]:
    example[method] = make_copies(X_chk[i:i + 1], y_chk[i:i + 1], method, np.random.default_rng(1))
for frame in range(2):
    print(f"\nExample Anger window, frame {frame + 1}")
    display(pd.DataFrame({k: v[0, frame, top] for k, v in example.items()},
                         index=[AU_NAMES[a] for a in top]).round(3))

## 8. Training and evaluation functions
- **Training:** early stopping on the domain-balanced validation macro-F1 (mean over the
  validation domains of the corpus), best weights restored.
- **Evaluation:** the test sets, once per run, with gate temperature 0.7.

Test metrics:
- *VR combined* = old VR + new VR test (607 windows, 7 people);
- *pooled* = all 3 test sets together;
- *domain-balanced* = mean of the 3 test sets;
- *false Neutral* = expressive VR windows called Neutral;
- *quiet accuracy* = accuracy on expressive VR windows with face size < 1.5.

In [ ]:
# ============================================================
# 8. TRAINING AND EVALUATION
# ============================================================

def macro_f1(y, pred):
    return f1_score(y, pred, labels=CLASS_LABELS, average="macro", zero_division=0)


class DomainBalancedMacroF1(callbacks.Callback):
    def __init__(self, val_domains):
        super().__init__()
        self.val_domains = val_domains

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        scores = {}
        for name, (X, y) in self.val_domains.items():
            scores[name] = macro_f1(y, self.model.predict(X, batch_size=256, verbose=0).argmax(1))
            logs[f"val_{name}_macro_f1"] = scores[name]
        logs["val_domain_balanced_macro_f1"] = float(np.mean(list(scores.values())))
        text = "  ".join(f"{k} {v:.4f}" for k, v in scores.items())
        print(f"  epoch {epoch + 1:2d}  {text}  domain-bal {logs['val_domain_balanced_macro_f1']:.4f}")


def run_tag(corpus, method, seed):
    return f"jitter_volume__{corpus}__{method.replace('+', '-')}__seed{seed}"


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def train_one(corpus, method, seed):
    c = CORPUS[corpus]
    tag = run_tag(corpus, method, seed)
    tf.keras.backend.clear_session()
    set_seed(seed)

    model, anneal_cb = build_moe_single_input(
        input_shape=(SEQ_LENGTH, N_AU), num_classes=len(CLASS_NAMES),
        tau_init=2.0, tau_final=0.7, gate_hidden=128, classifier_hidden=256,
        lb_coef=0.02, ent_coef=0.01,
    )
    data = AddCopies(c["X"], c["y"], c["w"], method, COPY_FRACTION, BATCH_SIZE, seed)
    X_val = np.concatenate([v[0] for v in c["val"].values()])
    y_val = np.concatenate([v[1] for v in c["val"].values()])

    print(f"\n=== {tag}: {len(data.X_epoch)} windows per epoch, {len(data)} steps")
    start = time.time()
    model.fit(
        data, validation_data=(X_val, y_val), epochs=EPOCHS, verbose=0,
        callbacks=[
            DomainBalancedMacroF1(c["val"]),
            callbacks.EarlyStopping(monitor="val_domain_balanced_macro_f1", mode="max",
                                    patience=PATIENCE, restore_best_weights=True),
            callbacks.CSVLogger(str(MODELS_DIR / f"{tag}_training_log.csv")),
            anneal_cb,
        ],
    )
    model.save(str(MODELS_DIR / f"{tag}.keras"))
    print(f"  trained in {(time.time() - start) / 60:.1f} min")
    return model


def evaluate_one(model, corpus, method, seed):
    tag = run_tag(corpus, method, seed)
    model.get_layer("gate").tau.assign(EVAL_GATE_TAU)

    log = pd.read_csv(MODELS_DIR / f"{tag}_training_log.csv")
    best = log.loc[log["val_domain_balanced_macro_f1"].idxmax()]
    row = {"corpus": corpus, "method": method, "seed": seed,
           "best epoch": int(best["epoch"]) + 1,
           "val domain-bal": best["val_domain_balanced_macro_f1"]}

    res = {k: (X, y, model.predict(X, batch_size=256, verbose=0)) for k, (X, y) in TEST.items()}
    res["VR combined"] = tuple(np.concatenate([res["old VR"][i], res["new VR"][i]]) for i in range(3))
    res["pooled"] = tuple(np.concatenate([res[k][i] for k in TEST]) for i in range(3))
    f1 = {k: macro_f1(y, p.argmax(1)) for k, (X, y, p) in res.items()}

    X, y, p = res["VR combined"]
    pred = p.argmax(1)
    expressive = y != NEUTRAL_ID
    quiet = expressive & (face_size(X) < 1.5)
    clipped = np.clip(p, 1e-12, 1)
    row.update({
        "test VR combined": f1["VR combined"],
        "test old VR": f1["old VR"],
        "test new VR": f1["new VR"],
        "test EmojiHero": f1["EmojiHero"],
        "test pooled": f1["pooled"],
        "test domain-bal": np.mean([f1[k] for k in TEST]),
        "VR correct": int(np.sum(pred == y)),
        "false Neutral": int(np.sum(expressive & (pred == NEUTRAL_ID))),
        "quiet accuracy": float(np.mean(pred[quiet] == y[quiet])),
        "log loss VR": log_loss(y, clipped / clipped.sum(1, keepdims=True), labels=CLASS_LABELS),
    })
    f1_emotion = precision_recall_fscore_support(y, pred, labels=CLASS_LABELS, zero_division=0)[2]
    emotion_row = {"corpus": corpus, "method": method, "seed": seed, **dict(zip(CLASS_NAMES, f1_emotion))}
    return row, emotion_row


def append_row(path, row):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not path.exists(), index=False)

## 9. Run all
Order: seed by seed, so that after the first seed you already have one full comparison. Finished
runs (already in `jitter_volume_results.csv`) are skipped, so this cell can be started again after a disconnect.

In [ ]:
# ============================================================
# 9. RUN ALL (resumable)
# ============================================================

def finished_runs():
    if not RESULTS_FILE.exists():
        return set()
    done = pd.read_csv(RESULTS_FILE)
    return {run_tag(r.corpus, r.method, r.seed) for r in done.itertuples()}


plan = [(corpus, method, seed) for seed in SEEDS for corpus in CORPORA for method in METHODS]
done = finished_runs()
todo = [p for p in plan if run_tag(*p) not in done]
print(f"{len(plan)} runs planned, {len(plan) - len(todo)} already done, {len(todo)} to do")

for k, (corpus, method, seed) in enumerate(todo, 1):
    print(f"\n[{k}/{len(todo)}]", end="")
    model = train_one(corpus, method, seed)
    row, emotion_row = evaluate_one(model, corpus, method, seed)
    append_row(RESULTS_FILE, row)
    append_row(EMOTION_FILE, emotion_row)
    print(f"  test VR combined {row['test VR combined']:.4f}  new VR {row['test new VR']:.4f}"
          f"  EmojiHero {row['test EmojiHero']:.4f}")

print("\nAll runs finished.")

## 10. Results
1. Mean ± standard deviation over the seeds.
2. Each augmentation vs `none` **with the same seed** (paired): mean change, seeds better/worse,
   paired t-test p.
3. Does jitter add something on top of volume? (`volume+jitter` vs `volume`)
4. F1 per emotion on VR combined.

With 5 seeds, read a change as real only if it goes the same way in (almost) every seed.

In [ ]:
# ============================================================
# 10. RESULTS
# ============================================================

results = pd.read_csv(RESULTS_FILE)
emotions = pd.read_csv(EMOTION_FILE)
METRICS = ["test VR combined", "test new VR", "test old VR", "test EmojiHero", "test pooled",
           "test domain-bal", "VR correct", "false Neutral", "quiet accuracy", "log loss VR",
           "val domain-bal"]
pd.set_option("display.width", 250)

# 1. Mean and standard deviation
for corpus in CORPORA:
    r = results[results.corpus == corpus]
    summary = r.groupby("method")[METRICS].agg(["mean", "std"]).reindex(METHODS)
    summary.columns = [f"{m} {s}" for m, s in summary.columns]
    print(f"\n{corpus}: mean and std over seeds (n seeds = {r.groupby('method').size().to_dict()})")
    display(summary.round(3).T)


# 2-3. Paired comparisons
def paired(corpus, method, reference):
    r = results[results.corpus == corpus]
    a = r[r.method == method].set_index("seed")
    b = r[r.method == reference].set_index("seed")
    seeds = a.index.intersection(b.index)
    out = []
    for m in METRICS:
        d = a.loc[seeds, m] - b.loc[seeds, m]
        p = ttest_rel(a.loc[seeds, m], b.loc[seeds, m]).pvalue if len(seeds) > 1 and d.std() > 0 else np.nan
        out.append({"metric": m, "mean change": d.mean(), "better": int((d > 0).sum()),
                    "worse": int((d < 0).sum()), "paired p": p})
    return pd.DataFrame(out).set_index("metric")


for corpus in CORPORA:
    for method, reference in [("volume", "none"), ("jitter", "none"),
                              ("volume+jitter", "none"), ("volume+jitter", "volume")]:
        print(f"\n{corpus}: {method} - {reference}  (same seed; for 'false Neutral' and "
              f"'log loss' lower is better)")
        display(paired(corpus, method, reference).round(3))

# 4. F1 per emotion
for corpus in CORPORA:
    e = emotions[emotions.corpus == corpus].groupby("method")[CLASS_NAMES].mean().reindex(METHODS)
    print(f"\n{corpus}: VR combined F1 per emotion, mean over seeds")
    display(e.round(3))